In [ ]:
import pandas as pd
from plotnine import *
import numpy as np
from vpop_calibration import *

%load_ext autoreload
%autoreload 2

## Loading Reference Data

In [ ]:
df = pd.read_csv("../../../examples/benchmarking/Mavoglurant/Mavoglurant_Dataset.csv")


obs_df = (
    df.loc[df["EVID"] == 0]
    .rename(columns={"ID": "id", "TIME": "time", "DV": "value", "DOSE": "dose"})[
        ["id", "time", "value", "dose", "WT"]
    ]
    .astype({"value": "float"})
)
obs_df["time"] = obs_df["time"].apply(lambda t: t * 60 * 60)
obs_df["value"] = obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "logC15"
obs_df["protocol_arm"] = "identity"
display(df.head())
display(obs_df.head())

## Reference model

In [ ]:
model = SimworkModelBinding(
    path_to_model="../../../examples/benchmarking/Mavoglurant/cm.json",
    path_to_solving_options="../../../examples/benchmarking/Mavoglurant/sv.json",
    inputs=["KbBR", "CLint", "KbMU", "KbAD", "KbBO", "KbRB", "dose", "WT"],
    outputs=["logC15"],
)
print(model.inputs)

struct_model = StructuralSimwork(model=model)

## Optimize model using SAEM

In [ ]:
prior_pdu = {
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 1},
        "KbBR": {"prior": np.exp(1.1), "prior_omega": 0.5},
        "KbMU": {"prior": np.exp(0.3), "prior_omega": 0.5},
        "KbBO": {"prior": np.exp(0.03), "prior_omega": 0.5},
        "KbAD": {"prior": np.exp(2), "prior_omega": 0.5},
        "KbRB": {"prior": np.exp(0.3), "prior_omega": 0.5},
    },
    "pdk": {"WT", "dose"},
    "error_model": {
        "logC15": {"error_type": "additive", "sigma": 0.5},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=0,
        nb_iter_learning=100,
        nb_iter_smoothing=100,
        plot_frames=5,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_model = NlmeModel(
    df=obs_df, prior_params=prior_pdu, structural_model=struct_model, config=config
)

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution()

In [ ]:
nlme_model.diagnostics.sampler.ebe_parameters_df.to_csv(
    "Mavoglurant_Simwork_ebe_PDU.csv", index=False
)

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(400)
nlme_model.diagnostics.compute_vpc(quantiles=[0.05, 0.5, 0.95])
vpc_py = nlme_model.diagnostics.vpc.copy()
vpc_py["model"] = "Simwork"
vpc_py.to_csv("Simwork_PDU_VPC.csv", index=False)
print(vpc_py.head(12))